# Inspect a knee MRI study (milestone 1)

Use this notebook to **manually** check series grouping, slice order, and plane labels.

Set `dataset_root` in `configs/data.yaml` to your DICOM archive. Reports are **not** used here.

If `data/raw` is empty, the last cell can build a **synthetic** study (geometry only, not clinical MRI) so you can exercise the viewer.


In [ ]:
from pathlib import Path

from orthovision import KneeMRIDataset, load_data_config
from orthovision.data.visualize import series_mosaic
import matplotlib.pyplot as plt

cfg = load_data_config()
print("dataset_root:", cfg.dataset_root)
print("exists:", cfg.dataset_root.exists())

ds = KneeMRIDataset(cfg)
try:
    stats = ds.scan()
    print(stats)
    ids = ds.study_ids()
    print("n_studies", len(ids))
except FileNotFoundError as e:
    print("Configure dataset_root:", e)
    ids = []


In [ ]:
# Set STUDY_ID to a real StudyInstanceUID, or leave None to pick the first scanned study.
STUDY_ID = None
N_SLICES = 7

if not ids:
    print("No studies discovered. Point dataset_root at DICOM files, or run the synthetic cell below.")
else:
    study_id = STUDY_ID or ids[0]
    summary = ds.study_summary(study_id)
    print("study", summary.study_id)
    print("series", summary.n_series, "valid", summary.n_valid_series, "warning", summary.n_warning_series, "invalid", summary.n_invalid_series)
    print("planes", summary.available_planes, "instances", summary.total_instances)
    for s in summary.series:
        print(f"  {s.series_id[-12:]}  {s.plane:8}  n={s.slice_count:3}  {s.rows}x{s.columns}  {s.validation_status}  {s.series_description}")
        print("   ", s.validation_messages)


In [ ]:
# Mosaic of evenly spaced ordered slices per series (VALID and WARNING).
# Check that anatomy progresses smoothly along the stack — that is the order QA.

if ids:
    study = ds.get_study(STUDY_ID or ids[0])
    for ser in study.series:
        status = ser.validation.status if ser.validation else "INVALID"
        if status == "INVALID":
            print("skip INVALID", ser.series_id)
            continue
        fig, _ = series_mosaic(ser, n_slices=N_SLICES)
        fig.suptitle(
            f"study={study.study_id[-16:]}  series={ser.series_id[-16:]}\n"
            f"{ser.series_description}  plane={ser.plane}  order={ser.ordering_audit.method}",
            fontsize=9,
        )
        plt.show()


In [ ]:
# Optional: synthetic geometry if the archive is empty (not a substitute for real MRI QA).
from pydicom.uid import generate_uid
from tests.synth_dicom import IOP_AXIAL, IOP_SAGITTAL, write_mr_slice
from orthovision.data.config import DataConfig
from orthovision.data.catalog import KneeMRIDataset

if not ids:
    root = Path("outputs") / "demo_synth" / "raw"
    study = generate_uid()
    sag, ax = generate_uid(), generate_uid()
    for i in range(6):
        write_mr_slice(root / "sag" / f"{i}.dcm", study_uid=study, series_uid=sag, instance_number=i + 1, ipp=(float(i), 0, 0), iop=IOP_SAGITTAL)
        write_mr_slice(root / "ax" / f"{i}.dcm", study_uid=study, series_uid=ax, instance_number=i + 1, ipp=(0, 0, float(i)), iop=IOP_AXIAL)
    demo_cfg = DataConfig(
        dataset_root=root.resolve(),
        output_dir=Path("outputs").resolve(),
        manifest_dir=Path("outputs/manifests").resolve(),
        viz_dir=Path("outputs/viz").resolve(),
        num_workers=1,
        logging_level="INFO",
        log_file=None,
        plane_min_alignment=0.8,
        plane_min_axis_separation=0.2,
        min_slices_warning=4,
        unusual_spacing_cv=0.15,
        duplicate_position_epsilon_mm=1e-3,
        orientation_epsilon=1e-3,
        project_root=Path(".").resolve(),
    )
    demo = KneeMRIDataset(demo_cfg)
    demo.scan()
    st = demo.get_study(study)
    print(demo.study_summary(study))
    for ser in st.series:
        series_mosaic(ser, n_slices=5)
        plt.show()
